# Milestone 1 — classification from your Drive input folder

**Ready model: ConvNeXt-Tiny, epoch 15. No retraining or manual path editing.**

1. Add the TA's images to **My Drive → Assignment_Live_Demo → milestone1 → input**.
2. In a new runtime, choose **Runtime → Run all** and approve the normal Drive connection when requested.
3. With the runtime prepared, upload more images to that folder and rerun only the last **Predict new images** cell.

The notebook scans subfolders too, preserves input photos, skips previously processed unchanged files and saves annotated results/tables/JSON/CSV into **milestone1/output** in Drive. Supplied test inputs are already in place. For another rehearsal of all images, use `run_demo(process_all=True)` in the last cell.

CPU is sufficient; a GPU is optional. Model weights are copied from Drive into the runtime cache and verified by SHA-256. These are predictions and can be wrong.


## 1. Install dependencies — once per fresh runtime


In [ ]:
import importlib.util, subprocess, sys
packages = ['Pillow==11.3.0', 'numpy==2.1.3', 'pandas>=2.2,<3']
if importlib.util.find_spec('torch') is None or importlib.util.find_spec('torchvision') is None:
    packages += ['torch>=2.5', 'torchvision>=0.20']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *packages], check=True)
print('Dependencies installed.')


## 2. Connect your Drive and configured folders


In [ ]:
from pathlib import Path
import importlib.util
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Assignment_Live_Demo')
MILESTONE = 1
MODEL_NAME = 'milestone1_convnext_tiny_best.pt'
EXPECTED_MODEL_SHA256 = 'fd5a80bd163b4c947aaa05d3749f4b7325e78cc6ef4017b2b18f93737c4c4c85'
FINAL_MAPPING = [{'folder': 'images_OBJ001', 'object_id': 'OBJ001', 'object_name': 'Basketball', 'class_index': 0}, {'folder': 'images_OBJ002', 'object_id': 'OBJ002', 'object_name': 'water bottle', 'class_index': 1}, {'folder': 'images_OBJ003', 'object_id': 'OBJ003', 'object_name': 'NoteBook', 'class_index': 2}, {'folder': 'images_OBJ004', 'object_id': 'OBJ004', 'object_name': 'Toy Car', 'class_index': 3}, {'folder': 'images_OBJ005', 'object_id': 'OBJ005', 'object_name': 'Computer mouse', 'class_index': 4}, {'folder': 'images_OBJ006', 'object_id': 'OBJ006', 'object_name': 'Capybara Toy', 'class_index': 5}, {'folder': 'images_OBJ007', 'object_id': 'OBJ007', 'object_name': 'Apple', 'class_index': 6}, {'folder': 'images_OBJ008', 'object_id': 'OBJ008', 'object_name': 'Cat', 'class_index': 7}, {'folder': 'images_OBJ009', 'object_id': 'OBJ009', 'object_name': 'Marshall Speaker', 'class_index': 8}, {'folder': 'images_OBJ010', 'object_id': 'OBJ010', 'object_name': 'Nike sneakers', 'class_index': 9}, {'folder': 'images_OBJ011', 'object_id': 'OBJ011', 'object_name': 'mug', 'class_index': 10}, {'folder': 'images_OBJ012', 'object_id': 'OBJ012', 'object_name': 'Computer Keyboard', 'class_index': 11}, {'folder': 'images_OBJ013', 'object_id': 'OBJ013', 'object_name': 'Mobile phone', 'class_index': 12}, {'folder': 'images_OBJ014', 'object_id': 'OBJ014', 'object_name': 'Sunglasses', 'class_index': 13}, {'folder': 'images_OBJ015', 'object_id': 'OBJ015', 'object_name': 'Water Tumbler', 'class_index': 14}, {'folder': 'images_OBJ016', 'object_id': 'OBJ016', 'object_name': 'Sandisk Portable SSD', 'class_index': 15}, {'folder': 'images_OBJ017', 'object_id': 'OBJ017', 'object_name': 'Controller', 'class_index': 16}, {'folder': 'images_OBJ018', 'object_id': 'OBJ018', 'object_name': 'Rubiks cube', 'class_index': 17}, {'folder': 'images_OBJ019', 'object_id': 'OBJ019', 'object_name': 'Plant', 'class_index': 18}, {'folder': 'images_OBJ020', 'object_id': 'OBJ020', 'object_name': 'Airpod', 'class_index': 19}, {'folder': 'images_OBJ021', 'object_id': 'OBJ021', 'object_name': 'Toy Bunny', 'class_index': 20}, {'folder': 'images_OBJ022', 'object_id': 'OBJ022', 'object_name': 'Guitar', 'class_index': 21}, {'folder': 'images_OBJ023', 'object_id': 'OBJ023', 'object_name': 'Bracelet', 'class_index': 22}, {'folder': 'images_OBJ024', 'object_id': 'OBJ024', 'object_name': 'Plush doll', 'class_index': 23}, {'folder': 'images_OBJ025', 'object_id': 'OBJ025', 'object_name': 'Watch', 'class_index': 24}, {'folder': 'images_OBJ026', 'object_id': 'OBJ026', 'object_name': 'Red Sox Cap', 'class_index': 25}, {'folder': 'images_OBJ027', 'object_id': 'OBJ027', 'object_name': 'Tennis ball', 'class_index': 26}, {'folder': 'images_OBJ028', 'object_id': 'OBJ028', 'object_name': 'banana', 'class_index': 27}, {'folder': 'images_OBJ029', 'object_id': 'OBJ029', 'object_name': 'Orange', 'class_index': 28}, {'folder': 'images_OBJ030', 'object_id': 'OBJ030', 'object_name': 'Tennis racket', 'class_index': 29}, {'folder': 'images_OBJ031', 'object_id': 'OBJ031', 'object_name': 'UNO card box', 'class_index': 30}, {'folder': 'images_OBJ032', 'object_id': 'OBJ032', 'object_name': 'Sound Card', 'class_index': 31}, {'folder': 'images_OBJ033', 'object_id': 'OBJ033', 'object_name': 'Duck', 'class_index': 32}, {'folder': 'images_OBJ034', 'object_id': 'OBJ034', 'object_name': 'Table Tennis Racket', 'class_index': 33}, {'folder': 'images_OBJ035', 'object_id': 'OBJ035', 'object_name': 'Badminton racket', 'class_index': 34}, {'folder': 'images_OBJ036', 'object_id': 'OBJ036', 'object_name': 'Pencil case', 'class_index': 35}, {'folder': 'images_OBJ037', 'object_id': 'OBJ037', 'object_name': 'Candle', 'class_index': 36}, {'folder': 'images_OBJ038', 'object_id': 'OBJ038', 'object_name': 'Perfume bottle', 'class_index': 37}, {'folder': 'images_OBJ039', 'object_id': 'OBJ039', 'object_name': 'Scissors', 'class_index': 38}, {'folder': 'images_OBJ040', 'object_id': 'OBJ040', 'object_name': 'Clutcher', 'class_index': 39}, {'folder': 'images_OBJ041', 'object_id': 'OBJ041', 'object_name': 'Table fan', 'class_index': 40}, {'folder': 'images_OBJ042', 'object_id': 'OBJ042', 'object_name': 'Mouth guard case', 'class_index': 41}, {'folder': 'images_OBJ043', 'object_id': 'OBJ043', 'object_name': 'Swimming Goggels', 'class_index': 42}, {'folder': 'images_OBJ044', 'object_id': 'OBJ044', 'object_name': 'Anime Figurine', 'class_index': 43}, {'folder': 'images_OBJ045', 'object_id': 'OBJ045', 'object_name': 'Wayfarer Sunglasses', 'class_index': 44}, {'folder': 'images_OBJ046', 'object_id': 'OBJ046', 'object_name': 'iWatch', 'class_index': 45}, {'folder': 'images_OBJ047', 'object_id': 'OBJ047', 'object_name': 'Backpack', 'class_index': 46}, {'folder': 'images_OBJ048', 'object_id': 'OBJ048', 'object_name': 'Wireless powerbank', 'class_index': 47}, {'folder': 'images_OBJ049', 'object_id': 'OBJ049', 'object_name': 'Pickleball bat', 'class_index': 48}, {'folder': 'images_OBJ050', 'object_id': 'OBJ050', 'object_name': 'Hairbrush', 'class_index': 49}, {'folder': 'images_OBJ051', 'object_id': 'OBJ051', 'object_name': 'tissue', 'class_index': 50}, {'folder': 'images_OBJ052', 'object_id': 'OBJ052', 'object_name': 'Spoon', 'class_index': 51}, {'folder': 'images_OBJ053', 'object_id': 'OBJ053', 'object_name': 'Moisturizer', 'class_index': 52}, {'folder': 'images_OBJ054', 'object_id': 'OBJ054', 'object_name': 'Purse', 'class_index': 53}, {'folder': 'images_OBJ055', 'object_id': 'OBJ055', 'object_name': 'Scientific Calculator', 'class_index': 54}, {'folder': 'images_OBJ056', 'object_id': 'OBJ056', 'object_name': "Trader Joe's Bag", 'class_index': 55}, {'folder': 'images_OBJ057', 'object_id': 'OBJ057', 'object_name': 'Power Bank', 'class_index': 56}, {'folder': 'images_OBJ058', 'object_id': 'OBJ058', 'object_name': 'Footstool', 'class_index': 57}, {'folder': 'images_OBJ059', 'object_id': 'OBJ059', 'object_name': 'Comb', 'class_index': 58}, {'folder': 'images_OBJ060', 'object_id': 'OBJ060', 'object_name': 'Sleep Mask', 'class_index': 59}, {'folder': 'images_OBJ061', 'object_id': 'OBJ061', 'object_name': 'Card wallet', 'class_index': 60}, {'folder': 'images_OBJ062', 'object_id': 'OBJ062', 'object_name': 'Headphones', 'class_index': 61}, {'folder': 'images_OBJ063', 'object_id': 'OBJ063', 'object_name': 'Measuring Bottle', 'class_index': 62}, {'folder': 'images_OBJ064', 'object_id': 'OBJ064', 'object_name': 'RGB Projector', 'class_index': 63}, {'folder': 'images_OBJ065', 'object_id': 'OBJ065', 'object_name': 'Analog watch', 'class_index': 64}, {'folder': 'images_OBJ066', 'object_id': 'OBJ066', 'object_name': 'hair oil', 'class_index': 65}, {'folder': 'images_OBJ067', 'object_id': 'OBJ067', 'object_name': "Men's Wallet", 'class_index': 66}, {'folder': 'images_OBJ068', 'object_id': 'OBJ068', 'object_name': 'Neck Pillow', 'class_index': 67}, {'folder': 'images_OBJ069', 'object_id': 'OBJ069', 'object_name': 'knife', 'class_index': 68}, {'folder': 'images_OBJ070', 'object_id': 'OBJ070', 'object_name': 'Astronaut projector', 'class_index': 69}, {'folder': 'images_OBJ071', 'object_id': 'OBJ071', 'object_name': 'Sloth Plushie', 'class_index': 70}, {'folder': 'images_OBJ072', 'object_id': 'OBJ072', 'object_name': 'Dog', 'class_index': 71}, {'folder': 'images_OBJ073', 'object_id': 'OBJ073', 'object_name': 'Bowl', 'class_index': 72}]
INPUT_DIR = DRIVE_ROOT / 'milestone1/input'
OUTPUT_DIR = DRIVE_ROOT / 'milestone1/output'
CONFIDENCE = 0.55
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError('Assignment_Live_Demo is missing from the connected My Drive. Check the Google account selected during mounting.')
print('TA images go here:',INPUT_DIR)
print('Results save here:',OUTPUT_DIR)


## 3. Model and folder-processing helpers


In [ ]:
"""ConvNeXt-Tiny inference matching the Milestone 1 controlled benchmark."""
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import textwrap
import uuid

import torch
from torch import nn
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image, ImageDraw, ImageFont, ImageOps

MEAN = (.485, .456, .406)
STD = (.229, .224, .225)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


@contextmanager
def temporary_cwd(directory):
    original = Path.cwd()
    Path(directory).mkdir(parents=True, exist_ok=True)
    try:
        os.chdir(directory)
        yield
    finally:
        os.chdir(original)


def upload_to(directory):
    from google.colab import files
    directory = Path(directory).resolve()
    with temporary_cwd(directory):
        uploaded = files.upload()
    paths = []
    for name in uploaded:
        path = (directory / name).resolve()
        if directory not in path.parents:
            raise ValueError('Unsafe upload filename')
        paths.append(path)
    return paths


def verify_model_file(path, expected_sha256):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Model file not found: {path}')
    if sha256_file(path) != expected_sha256:
        raise ValueError('This is not our selected classification checkpoint. '
                         'Upload milestone1_convnext_tiny_best.pt supplied with this notebook.')
    return path


class LiveClassifier(nn.Module):
    """Exact ConvNeXt branch and common classifier head used in training."""
    def __init__(self, classes=73):
        super().__init__()
        self.backbone = models.convnext_tiny(weights=None)
        features = self.backbone.classifier[2].in_features
        self.backbone.classifier[2] = nn.Identity()
        self.head = nn.Sequential(nn.Linear(features, 256), nn.LayerNorm(256),
                                  nn.GELU(), nn.Dropout(.3), nn.Linear(256, classes))

    def forward(self, images):
        return self.head(self.backbone(images))


def load_classifier(path, expected_sha256, mapping, device='cpu'):
    path = verify_model_file(path, expected_sha256)
    # Read only the exact trusted checkpoint whose hash was verified above.
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)
    if checkpoint.get('architecture') != 'convnext_tiny' or checkpoint.get('epoch') != 15:
        raise ValueError('Checkpoint architecture/epoch differs from the selected experiment')
    if checkpoint.get('classes') != mapping or len(mapping) != 73:
        raise ValueError('Checkpoint classes differ from the final Object IDs/names')
    if [m['class_index'] for m in mapping] != list(range(73)):
        raise ValueError('Invalid final class mapping')
    model = LiveClassifier(len(mapping))
    model.load_state_dict(checkpoint['model'], strict=True)
    model.to(device).eval()
    return model


def evaluation_transform():
    return transforms.Compose([
        transforms.Resize(256, interpolation=InterpolationMode.BILINEAR, antialias=True),
        transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


def result_card(image, prediction):
    width = max(600, min(1000, image.width))
    preview = ImageOps.contain(image, (width, 650), method=Image.Resampling.BILINEAR)
    try:
        font = ImageFont.truetype('DejaVuSans.ttf', 26)
        small = ImageFont.truetype('DejaVuSans.ttf', 20)
    except OSError:
        font, small = ImageFont.load_default(size=26), ImageFont.load_default(size=20)
    lines = textwrap.wrap(prediction['object_name'], width=max(20, int((width-32)/16)))
    header = 104 + 34*len(lines)
    panel = Image.new('RGB', (width, header+preview.height+16), 'white')
    draw = ImageDraw.Draw(panel)
    draw.text((16, 12), f"Predicted Object ID: {prediction['object_id']}", font=font, fill='#005c61')
    for index, line in enumerate(lines):
        draw.text((16, 48+34*index), line, font=font, fill='#162f46')
    draw.text((16, header-36), f"Model score: {prediction['score']:.2%}", font=small, fill='#516474')
    panel.paste(preview, ((width-preview.width)//2, header))
    return panel


def classify_image(model, image_path, mapping, output, device='cpu', model_sha256=None, top_k=5):
    image_path, output = Path(image_path), Path(output)
    output.mkdir(parents=True, exist_ok=True)
    with Image.open(image_path) as original:
        original.seek(0)
        image = ImageOps.exif_transpose(original).convert('RGB')
    tensor = evaluation_transform()(image).unsqueeze(0).to(device)
    with torch.inference_mode():
        scores = model(tensor).softmax(1)[0].cpu()
    if scores.numel() != len(mapping) or not torch.isfinite(scores).all():
        raise ValueError('Invalid classifier output')
    values, indices = scores.topk(min(top_k, len(mapping)))
    ranked = [{'rank': rank, 'object_id': mapping[index]['object_id'],
               'object_name': mapping[index]['object_name'], 'score': float(score)}
              for rank, (score, index) in enumerate(zip(values.tolist(), indices.tolist()), 1)]
    record = {'input_filename': image_path.name, 'input_sha256': sha256_file(image_path),
              'model_sha256': model_sha256, 'architecture': 'convnext_tiny', 'selected_epoch': 15,
              'width': image.width, 'height': image.height, 'prediction': ranked[0], 'top_k': ranked,
              'score_type': 'softmax model score; not a calibrated correctness probability',
              'preprocessing': {'orientation': 'EXIF-oriented primary RGB photo',
                                'resize_short_side': 256, 'center_crop': [224, 224],
                                'mean': list(MEAN), 'std': list(STD)},
              'generated_utc': datetime.now(timezone.utc).isoformat()}
    prefix = image_path.stem+'_'+uuid.uuid4().hex[:8]
    card = output/f'{prefix}_classification.png'
    result_card(image, ranked[0]).save(card)
    (output/f'{prefix}_classification.json').write_text(json.dumps(record, indent=2))
    with (output/f'{prefix}_classification.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=['rank', 'object_id', 'object_name', 'score'])
        writer.writeheader()
        writer.writerows(ranked)
    return record, ranked, card


"""Drive-folder inference runner embedded in both self-contained Colab demos."""
import hashlib
import json
from pathlib import Path
from datetime import datetime, timezone
import uuid


def write_state(path, value):
    path = Path(path)
    temporary = path.with_name(path.name+'.temporary-'+uuid.uuid4().hex[:8])
    temporary.write_text(json.dumps(value, indent=2))
    temporary.replace(path)


def prepare_cached_model(source, destination, expected_hash):
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and sha256_file(destination)==expected_hash:
        return destination
    if not source.is_file():
        raise FileNotFoundError(f'Model missing from configured Drive path: {source}')
    import shutil
    temporary = destination.with_suffix('.partial')
    shutil.copyfile(source, temporary)
    if sha256_file(temporary)!=expected_hash:
        raise ValueError('Checkpoint checksum differs from the selected experiment')
    temporary.replace(destination)
    return destination


def run_demo(process_all=False):
    """Find new/changed input photos, predict and persist results without moving inputs."""
    import pandas as pd
    from IPython.display import display, Image as DisplayImage
    supported = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff','.mpo'}
    INPUT_DIR.mkdir(parents=True,exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    state_path = OUTPUT_DIR/'processed_inputs.json'
    state = json.loads(state_path.read_text()) if state_path.exists() else {'schema':1,'processed':{}}
    signature = EXPECTED_MODEL_SHA256 + (f':confidence={CONFIDENCE}' if MILESTONE==2 else ':eval256crop224')
    pending, old = [], 0
    candidates = sorted(p for p in INPUT_DIR.rglob('*') if p.is_file() and p.suffix.lower() in supported)
    for image in candidates:
        relative = image.relative_to(INPUT_DIR).as_posix()
        digest = sha256_file(image)
        previous = state['processed'].get(relative)
        if not process_all and previous and previous['sha256']==digest and previous['signature']==signature:
            old += 1
            continue
        pending.append((image,relative,digest))
    print(f'Input folder: {INPUT_DIR}')
    print(f'{len(pending)} new/changed image(s); {old} unchanged image(s) already processed.')
    if not pending:
        print('Upload the TA photos into the input folder, wait for the Drive upload to finish, then rerun this cell.')
        print('Previous results remain in:', OUTPUT_DIR)
        print('To repeat all practice images, run run_demo(process_all=True).')
        return {'processed':0,'skipped':old,'errors':[]}
    output = OUTPUT_DIR/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:6])
    output.mkdir(parents=True)
    records, errors = [], []
    for image,relative,digest in pending:
        try:
            if MILESTONE==1:
                record,rows,preview = classify_image(active_model,image,FINAL_MAPPING,output,
                    device=DEVICE,model_sha256=EXPECTED_MODEL_SHA256)
                top = record['prediction']
                print(f"{relative}: {top['object_id']} — {top['object_name']} (score {top['score']:.2%})")
            else:
                record,rows,preview = detect_image(active_model,image,FINAL_MAPPING,output,
                    confidence=CONFIDENCE,device=DEVICE,model_sha256=EXPECTED_MODEL_SHA256)
                print(f'{relative}: {len(rows)} predicted object(s), confidence ≥ {CONFIDENCE:.2f}')
            if sha256_file(image)!=digest:
                raise RuntimeError('Input changed during prediction; wait for the upload to finish and rerun')
            record['drive_input_relative_path'] = relative
            records.append(record)
            display(DisplayImage(filename=str(preview),width=min(900,record['width']) if MILESTONE==2 else 700))
            if rows:
                display(pd.DataFrame(rows).round({'confidence':3} if MILESTONE==2 else {'score':4}))
            else:
                print('No detections above the threshold. The model can miss objects.')
            state['processed'][relative] = {'sha256':digest,'signature':signature,
                'result_folder':output.name,'preview':preview.relative_to(OUTPUT_DIR).as_posix()}
            write_state(state_path,state)
        except Exception as error:
            errors.append({'input':relative,'error':str(error)})
            print(f'Could not process {relative}: {error}. It will be retried next time.')
    write_state(output/'batch_results.json',{'milestone':MILESTONE,'model_sha256':EXPECTED_MODEL_SHA256,
                 'records':records,'errors':errors})
    write_state(OUTPUT_DIR/'latest_run.json',{'result_folder':output.name,'processed':len(records),'errors':errors})
    print('Results saved permanently in Drive:',output)
    return {'processed':len(records),'skipped':old,'errors':errors,'output':str(output)}


## 4. Load the saved checkpoint — reused while the runtime stays connected


In [ ]:
import torch
cache = Path('/content/assignment_live_demo_cache') if IN_COLAB else DRIVE_ROOT.parent/'runtime_model_cache'
cached_path = prepare_cached_model(DRIVE_ROOT/'models'/MODEL_NAME,cache/MODEL_NAME,EXPECTED_MODEL_SHA256)
model_key = (EXPECTED_MODEL_SHA256, 'cuda' if torch.cuda.is_available() else 'cpu')
if globals().get('active_model_key')!=model_key:
    if MILESTONE==1:
        DEVICE = model_key[1]
        active_model = load_classifier(cached_path,EXPECTED_MODEL_SHA256,FINAL_MAPPING,DEVICE)
    else:
        from ultralytics import YOLO
        DEVICE = '0' if torch.cuda.is_available() else 'cpu'
        active_model = YOLO(str(cached_path))
        check_object_ids(active_model,FINAL_MAPPING)
    active_model_key = model_key
print('Ready:',MODEL_NAME,'device:',DEVICE)


## 5. Predict new images — rerun this cell during the demo
Wait until Drive says the upload is complete, then press ▶ on the code cell below. Results appear here and in the Drive output folder. If you add more images while the notebook is open, rerun this cell; there is no need to reload the model.

To repeat already-processed practice inputs, change the call to `run_demo(process_all=True)`. Do not manually alter predicted IDs or boxes. The existing confidence and preprocessing match the reported experiments.


In [ ]:
summary = run_demo()


If the runtime disconnects or restarts, use **Run all** again. If no new images are found, check the upload finished and that you used this milestone's input folder. Old predictions remain in the dated output folders. The source dataset is not edited.
